In [ ]:
# Do not use now
# Block-1
!pip install vllm --upgrade

In [ ]:
# Do not use now
# Block-2

import os
import subprocess
import time

# Define the exact model and parameters
MODEL_NAME = "nvidia/NVIDIA-Nemotron-Labs-Dory1.0-4B-MXFP8"

print("Starting vLLM server with Dory in the background...")

# Configure clean environments for stable legacy engine execution
env = os.environ.copy()
env["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

# Use subprocess to run 'vllm serve' explicitly forcing the legacy stable backend to avoid the V1 engine core handshake bugs
vllm_process = subprocess.Popen([
    "python3", "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_NAME,
    "--port", "8000",
    "--host", "0.0.0.0",
    "--trust-remote-code",
    "--gpu-memory-utilization", "0.80",
    "--tensor-parallel-size", "1",
    "--v1", "False"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)

# Give the model a couple of minutes to download weights and allocate VRAM
print("Waiting for server boot sequence... (This takes 1-2 minutes on first run)")
time.sleep(90)

print("Server container initiated! Proceed to run your Python code review cell.")

### Running vLLM on CPU

To run the server utilizing CPU instead of GPU, we pass `--device cpu` and specify `--dtype float32` (or keep it default/bfloat16 depending on support) to avoid precision errors on non-GPU hardware.

In [45]:
# Do not use now
# Block-3
import os
import subprocess
import time

MODEL_NAME = "nvidia/NVIDIA-Nemotron-Labs-Dory1.0-4B-MXFP8"

print("Starting vLLM server on CPU in the background...")

env = os.environ.copy()
# Force vLLM to target the CPU backend
env["VLLM_TARGET_DEVICE"] = "cpu"
# Hide CUDA devices entirely to prevent PyTorch from initializing CUDA
env["CUDA_VISIBLE_DEVICES"] = ""

vllm_cpu_process = subprocess.Popen([
    "python3", "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_NAME,
    "--port", "8000",
    "--host", "0.0.0.0",
    "--trust-remote-code",
    "--dtype", "float32"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)

print("CPU Server process spawned successfully in the background!")
print("Please wait 1-2 minutes for the model to download and load into system RAM, then run the health check cell.")

Starting vLLM server on CPU in the background...
CPU Server process spawned successfully in the background!
Please wait 1-2 minutes for the model to download and load into system RAM, then run the health check cell.


In [46]:
# Do not use now
# Block-4
import time, socket

def wait_for_server(port=8000, timeout=120):
  start = time.time()
  while time.time() - start < timeout:
    try:
      with socket.create_connection(('127.0.0.1', port), timeout=1):
        print('Server is up!')
        return True
    except OSError:
      time.sleep(2)
  print('Server timeout!')
  return False


wait_for_server(8000)

Server timeout!


False

In [44]:
# Do not use now
# Block-5
!fuser 8000/tcp

In [41]:
# Do not use now
# Block-6
import requests

try:
    response = requests.get("http://localhost:8000/v1/models", timeout=5)
    print("Status Code:", response.status_code)
    print("Response:", response.json())
except Exception as e:
    print("Health check failed:", e)

Health check failed: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /v1/models (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7e0bbb97ae00>: Failed to establish a new connection: [Errno 111] Connection refused'))


In [ ]:
# Do not use now
# Block-7
if 'vllm_cpu_process' in globals():
    poll = vllm_cpu_process.poll()
    print(f"Current Process State: {'Terminated' if poll is not None else 'Running'}")
    if poll is not None:
        print(f"Exit Code: {poll}")

    print("--- Latent Output Logs (Stdout/Stderr) ---")
    try:
        # Read all available lines from the non-blocking pipe
        import os
        import fcntl
        fd = vllm_cpu_process.stdout.fileno()
        fl = fcntl.fcntl(fd, fcntl.F_GETFL)
        fcntl.fcntl(fd, fcntl.F_SETFL, fl | os.O_NONBLOCK)

        logs = vllm_cpu_process.stdout.read()
        if logs:
            print(logs)
        else:
            print("No logs buffered at this moment.")
    except Exception as e:
        print("Could not read process streams:", e)
else:
    print("vllm_cpu_process not found in globals.")

In [ ]:
# Do not use now
# Block-8
!pip install openai

In [ ]:
# Do not use now
# Block-9
import requests
from openai import OpenAI

# Quick diagnostics: Check if the port is open and reachable
try:
    response = requests.get("http://localhost:8000/v1/models", timeout=3)
    print("✅ Connected to vLLM server successfully!")
except Exception as e:
    print("❌ Connection to http://localhost:8000 failed.")
    print("The background server might still be loading, or it crashed during boot.\n")

    # Try to check on the global process if defined in the notebook context
    if 'vllm_process' in globals():
        poll = vllm_process.poll()
        if poll is not None:
            print(f"⚠️ The vLLM server process has terminated with exit code {poll}.")
            print("Latest logs from the server process:")
            try:
                # Safely read buffered stdout without triggering I/O on closed file error
                if vllm_process.stdout and not vllm_process.stdout.closed:
                    stdout = vllm_process.stdout.read()
                    print(stdout)
                else:
                    # Fallback to the global stdout variable cached during the session
                    if 'stdout' in globals():
                        print(stdout[-2000:]) # Print last 2000 characters of the captured log
                    else:
                        print("I/O streams are closed and logs could not be retrieved.")
            except Exception as io_err:
                print(f"Could not read process streams: {io_err}")
        else:
            print("🕒 The server process is still running but not responding yet. Try waiting another 60 seconds.")
    else:
        print("⚠️ Could not find 'vllm_process' in global variables. Please re-run the startup cell.")

# Initialize client pointing to your local vLLM server running in Colab
client = OpenAI(
    base_url="http://localhost:8000/v1",  # Local vLLM server port
    api_key="EMPTY"  # vLLM doesn't require a key by default
)

# Specify the model matching the server setup
MODEL_NAME = "nvidia/NVIDIA-Nemotron-Labs-Dory1.0-4B-MXFP8"

# Simple Python function with bad practices for testing the POC
code_to_review = """
def process(data):
    # bad variable naming, no docstring, missing type hints
    x = []
    for item in data:
        try:
            # Catch-all exception and nested mutation
            if item.get("status") == "active":
                x.append(item["value"] * 2)
        except:
            pass
    return x
"""

# Construct the prompt for Dory
prompt = f"""You are an expert Python code reviewer. Look at the following Python function and perform a concise code review.
Identify structural issues, styling concerns (PEP 8), and potential bugs. Suggest improvements.

```python
{code_to_review}
```
"""

print("\nSending code to Dory for review...\n")

try:
    # Request code review from Dory using streaming for real-time output
    stream = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1024,
        temperature=0.2, # Lower temperature for analytical tasks
        stream=True
    )

    # Print the streamed response token-by-token
    for chunk in stream:
        content = chunk.choices[0].delta.content or ""
        print(content, end="", flush=True)
except Exception as e:
    print(f"❌ Failed to generate review: {e}")

In [ ]:
# Do not use now
# Block-201
#!pip install -U transformers torch accelerate

In [ ]:
# Do not use now
# Block-202
!pip install git+https://github.com/huggingface/transformers.git

In [ ]:
# Do not use now
# Block-203
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "nvidia/NVIDIA-Nemotron-Labs-Dory1.0-4B-BF16"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)



In [ ]:
# Do not use now
# Block-204
# Example generation
input_text = "Explain what a looped latent-recurrent language model is in simple terms:"
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
# use now
# Block-301
!pip install vllm --upgrade

In [ ]:
# Do not use now
# Block-302

!python -m vllm.entrypoints.openai.api_server \
    --model nvidia/NVIDIA-Nemotron-Labs-Dory1.0-4B-MXFP8 \
    --port 8000

In [ ]:
# Do not use now
# Block-303
import os
import subprocess
import sys

# Set stability flags for the T4 GPU
os.environ["VLLM_USE_V1"] = "0"

MODEL_NAME = "nvidia/NVIDIA-Nemotron-Labs-Dory1.0-4B-MXFP8"

print("🚀 Launching Dory server in the background...")

# Start the process
vllm_process = subprocess.Popen([
    "vllm", "serve", MODEL_NAME,
    "--port", "8000",
    "--host", "0.0.0.0",
    "--gpu-memory-utilization", "0.85",
    "--max-model-len", "2048",
    "--enforce-eager",
    "--disable-custom-all-reduce"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

# Monitor the logs line by line
print("⏳ Loading weights and initializing the engine (monitoring logs)...")

ready = False
while True:
    line = vllm_process.stdout.readline()
    if not line:
        break

    # Print the logs in real-time so you can see the progress
    sys.stdout.write(line)
    sys.stdout.flush()

    # Check for the magic string that means the server is online
    if "Uvicorn running on" in line or "Application startup complete" in line:
        ready = True
        break

    # Check if the process crashed early
    if vllm_process.poll() is not None:
        print("\n❌ Server failed to start. Check the logs above for the error details.")
        break

if ready:
    print("\n" + "="*60)
    print("🎉 SUCCESS: Dory weights are fully loaded!")
    print("🔌 The server is up and running. You can now execute your code review cell.")
    print("="*60)


In [ ]:
# do not use now
# Block-304
!pip install openai


In [ ]:
# do not use now
# Block-305
from openai import OpenAI

# Initialize client pointing to your local vLLM server running in Colab
client = OpenAI(
    base_url="http://localhost:8000/v1",  # Local vLLM server port
    api_key="EMPTY"  # vLLM doesn't require a key by default
)

# Specify the model matching the server setup
MODEL_NAME = "nvidia/NVIDIA-Nemotron-Labs-Dory1.0-4B-MXFP8"

# Simple Python function with bad practices for testing the POC
code_to_review = """
def process(data):
    # bad variable naming, no docstring, missing type hints
    x = []
    for item in data:
        try:
            # Catch-all exception and nested mutation
            if item.get("status") == "active":
                x.append(item["value"] * 2)
        except:
            pass
    return x
"""

# Construct the prompt for Dory
prompt = f"""You are an expert Python code reviewer. Look at the following Python function and perform a concise code review.
Identify structural issues, styling concerns (PEP 8), and potential bugs. Suggest improvements.

```python
{code_to_review}
```
"""

print("Sending code to Dory for review...\n")

# Request code review from Dory using streaming for real-time output
stream = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": prompt}],
    max_tokens=1024,
    temperature=0.2, # Lower temperature for analytical tasks
    stream=True
)

# Print the streamed response token-by-token
for chunk in stream:
    content = chunk.choices[0].delta.content or ""
    print(content, end="", flush=True)


In [ ]:
# use now
# Block-306
!pip install vllm --upgrade

In [ ]:
# Do not use now
# Block-307
import os
from vllm import LLM, SamplingParams

# Force older backend constraints safely inside the engine
os.environ["VLLM_ATTENTION_BACKEND"] = "XFORMERS"

MODEL_NAME = "nvidia/NVIDIA-Nemotron-Labs-Dory1.0-4B-MXFP8"

print("⏳ Loading weights directly into native Python memory...")

# Instantiate the model directly without the V1 API background core
llm = LLM(
    model=MODEL_NAME,
    gpu_memory_utilization=0.85, # Leaves memory room on T4's 15GB VRAM
    max_model_len=2048,          # Restricts context to avoid OOM
    enforce_eager=True           # Disables Triton's crashing CUDA graph profiles
)

# ----------------------------------------------------
# Define your POC Code Review Task
# ----------------------------------------------------
code_to_review = """
def process(data):
    # bad variable naming, no docstring, missing type hints
    x = []
    for item in data:
        try:
            # Catch-all exception and nested mutation
            if item.get("status") == "active":
                x.append(item["value"] * 2)
        except:
            pass
    return x
"""

prompt = f"""You are an expert Python code reviewer. Look at the following Python function and perform a concise code review.
Identify structural issues, styling concerns (PEP 8), and potential bugs. Suggest improvements.

```python
{code_to_review}
```
"""

# Configure standard generation constraints
sampling_params = SamplingParams(
    temperature=0.2,
    max_tokens=1024
)

print("\n🤖 Running code review loop through Dory...")

# Execute generation directly
outputs = llm.generate([prompt], sampling_params)

# Print the result
for output in outputs:
    generated_text = output.outputs[0].text
    print("\n=== Dory's Code Review ===")
    print(generated_text)


In [ ]:
# do not use now
# Block-308
import os

# Crucial: Must be exactly uppercase "CPU" to match vLLM's internal platform checker
os.environ["VLLM_TARGET_DEVICE"] = "CPU"

# Import vLLM ONLY after the environment variable is set
from vllm import LLM, SamplingParams

# Using the optimized Dory GGUF matrix variant for CPU RAM safety
MODEL_NAME = "nvidia/NVIDIA-Nemotron-Labs-Dory1.0-4B-GGUF"

print("⏳ Initializing Dory GGUF natively on the CPU platform...")

# Initialize the engine
llm = LLM(
    model=MODEL_NAME,
    max_model_len=1024,  # Tight context window to remain safe under standard CPU RAM limits
)

# ----------------------------------------------------
# POC Code Review Target Definition
# ----------------------------------------------------
code_to_review = """
def process(data):
    # bad variable naming, no docstring, missing type hints
    x = []
    for item in data:
        try:
            # Catch-all exception and nested mutation
            if item.get("status") == "active":
                x.append(item["value"] * 2)
        except:
            pass
    return x
"""

prompt = f"""You are an expert Python code reviewer. Look at the following Python function and perform a concise code review.
Identify structural issues, styling concerns (PEP 8), and potential bugs. Suggest improvements.

```python
{code_to_review}
```
"""

sampling_params = SamplingParams(
    temperature=0.2,
    max_tokens=512
)

print("\n🤖 Running code review loop through Dory on CPU...")

# Generate output
outputs = llm.generate([prompt], sampling_params)

# Print the final output
for output in outputs:
    print("\n=== Dory's Code Review ===")
    print(output.outputs.text)


In [ ]:
# use now
# Block-401
!pip install llama-cpp-python openai

In [9]:
# use now
# Block-402

from huggingface_hub import hf_hub_download

REPO_ID = "nvidia/NVIDIA-Nemotron-Labs-Dory1.0-4B-GGUF"
# Crucial: Fixed exact casing for Q5_K_M.gguf
FILENAME = "NVIDIA-Nemotron-Labs-Dory1.0-4B-Q5_K_M.gguf"

print("⏳ Downloading verified Dory 4B GGUF file from Hugging Face...")
try:
    model_path = hf_hub_download(
        repo_id=REPO_ID,
        filename=FILENAME
    )
    print(f"\n✅ Success! File downloaded and cached at:\n{model_path}")
except Exception as e:
    print(f"\n❌ Download failed. Error Details: {e}")



⏳ Downloading verified Dory 4B GGUF file from Hugging Face...


NVIDIA-Nemotron-Labs-Dory1.0-4B-Q5_K_M.g(…): reconstructing file:   0%|          |  0.00B / 3.67GB            

NVIDIA-Nemotron-Labs-Dory1.0-4B-Q5_K_M.g(…): downloading bytes:           |  0.00B            


✅ Success! File downloaded and cached at:
/root/.cache/huggingface/hub/models--nvidia--NVIDIA-Nemotron-Labs-Dory1.0-4B-GGUF/snapshots/df53755cb8741843af8e019e088819fbef17e29b/NVIDIA-Nemotron-Labs-Dory1.0-4B-Q5_K_M.gguf


In [10]:
# use now
# Block-403
import subprocess

print("🚀 Launching Dory CPU Server on port 8000...")

# Start the CPU server in the background
server_process = subprocess.Popen([
    "python3", "-m", "llama_cpp.server",
    "--model", model_path,
    "--port", "8000",
    "--n_ctx", "2048"  # Set context buffer size for processing code
])

print("🔌 Server started! Move to the next cell to run your code review request.")


🚀 Launching Dory CPU Server on port 8000...
🔌 Server started! Move to the next cell to run your code review request.


In [ ]:
# do not use now
# Block-404
import sys
from llama_cpp import Llama

# 'model_path' carries over from your successful download in the previous step
print("⏳ Initializing Dory GGUF directly in Python memory...")

# Load the model directly without a webserver layer
llm = Llama(
    model_path=model_path,
    n_ctx=2048,      # Context window size for code processing
    n_threads=2,     # Match typical Colab CPU tier configuration
    verbose=False    # Suppresses messy C++ log outputs
)

# Define the target function for the POC code review
code_to_review = """
def process(data):
    # bad variable naming, no docstring, missing type hints
    x = []
    for item in data:
        try:
            # Catch-all exception and nested mutation
            if item.get("status") == "active":
                x.append(item["value"] * 2)
        except:
            pass
    return x
"""

# Construct the prompt structure
prompt = f"""You are an expert Python code reviewer. Look at the following Python function and perform a concise code review.
Identify structural issues, styling concerns (PEP 8), and potential bugs. Suggest improvements.

```python
{code_to_review}
```
"""

print("\n🤖 Running code review via native Dory pipeline...\n")

# Execute generation directly
response = llm(
    prompt,
    max_tokens=512,
    temperature=0.2
)

# Extract and display the markdown text response
review_text = response["choices"][0]["text"]
print("=== Dory's Code Review ===")
print(review_text)


In [ ]:
#  use now
# Block-405
# 1. Clone the specific pinned commit fork supporting Dory's recurrent layers
!git clone https://github.com/ggml-org/llama.cpp.git
%cd llama.cpp
# Fetch and check out the Dory implementation branch
!git fetch origin pull/30306/head:dory-branch
!git checkout dory-branch

# 2. Build the C++ binaries from source directly inside Colab
!cmake -B build
!cmake --build build -j --target llama-cli

# 3. Define the prompt string directly to feed the custom executable
import json
prompt_str = "You are an expert Python code reviewer. Look at this function and review it: def process(data): x=[]; for item in data: x.append(item) return x"

# 4. Invoke the custom compiled terminal tool passing your downloaded GGUF file
# (Replace the path string with your exact downloaded file location)
!./build/bin/llama-cli -m "/root/.cache/huggingface/hub/models--nvidia--NVIDIA-Nemotron-Labs-Dory1.0-4B-GGUF/snapshots/df53755cb8741843af8e019e088819fbef17e29b/NVIDIA-Nemotron-Labs-Dory1.0-4B-Q5_K_M.gguf" \
    -p "{prompt_str}" \
    -n 512 \
    --threads 2


In [2]:
#  do not use now
# Block-501
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Use the base high-precision repository that doesn't rely on GGUF layout matrix modifications
MODEL_NAME = "nvidia/NVIDIA-Nemotron-Labs-Dory1.0-4B-MXFP8"

print("⏳ Initializing Dory directly onto the CPU... (This will download weights, please hold)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="cpu",               # Forces the model explicitly onto system CPU cores
    torch_dtype=torch.float32,     # Standard format safe for pure CPU execution arithmetic
    trust_remote_code=True          # Required to run Dory's custom looped layer calculations
)

# ----------------------------------------------------
# POC Code Review Target Definition
# ----------------------------------------------------
code_to_review = """
def process(data):
    # bad variable naming, no docstring, missing type hints
    x = []
    for item in data:
        try:
            # Catch-all exception and nested mutation
            if item.get("status") == "active":
                x.append(item["value"] * 2)
        except:
            pass
    return x
"""

prompt = f"""You are an expert Python code reviewer. Look at the following Python function and perform a concise code review.
Identify structural issues, styling concerns (PEP 8), and potential bugs. Suggest improvements.

```python
{code_to_review}
```
"""

# Format input tensors using the model's text pipeline
inputs = tokenizer(prompt, return_tensors="pt").to("cpu")

print("\n🤖 Running code review sequence through Dory on CPU...")

# Generate text response sequentially
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.2,
        do_sample=True
    )

# Decode and print results
review_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n=== Dory's Code Review ===")
print(review_text[len(prompt):]) # Slices prompt to only show the model's feedback


⏳ Initializing Dory directly onto the CPU... (This will download weights, please hold)


config.json:   0%|          | 0.00/16.9k [00:00<?, ?B/s]

[transformers] You are using a model of type `dory` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


StrictDataclassClassValidationError: Class validation error for validator 'validate_layer_type':
    ValueError: The `layer_types` entries must be in ('full_attention', 'sliding_attention', 'chunked_attention', 'window_attention', 'indexed_attention', 'compressed_sparse_attention', 'heavily_compressed_attention', 'minimax_m3_sparse', 'conv', 'moe', 'hybrid', 'hybrid_sliding', 'reference_sliding_attention', 'linear_attention') but got ['sliding_attention', 'mlp', 'sliding_attention', 'mlp', 'sliding_attention', 'mlp', 'full_attention', 'mlp', 'sliding_attention', 'mlp', 'sliding_attention', 'mlp', 'sliding_attention', 'mlp', 'full_attention', 'mlp', 'sliding_attention', 'mlp', 'sliding_attention', 'mlp', 'sliding_attention', 'mlp', 'full_attention', 'mlp', 'sliding_attention', 'mlp', 'sliding_attention', 'mlp', 'sliding_attention', 'mlp', 'full_attention', 'mlp', 'sliding_attention', 'mlp', 'sliding_attention', 'mlp', 'sliding_attention', 'mlp', 'full_attention', 'mlp', 'sliding_attenti

In [ ]:
# use now
# Block-601
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

# 1. Fetch the verified GGUF from a reliable public mirror
print("⏳ Fetching standard architecture Nemotron-Mini 4B GGUF...")
try:
    nemotron_path = hf_hub_download(
        repo_id="bartowski/Nemotron-Mini-4B-Instruct-GGUF",
        filename="Nemotron-Mini-4B-Instruct-Q5_K_M.gguf" # Recommended 5-bit quantization
    )
    print(f"✅ Download complete! File stored at: {nemotron_path}")
except Exception as e:
    print(f"❌ Mirror 1 failed, trying fallback repository...")
    nemotron_path = hf_hub_download(
        repo_id="QuantFactory/Nemotron-Mini-4B-Instruct-GGUF",
        filename="Nemotron-Mini-4B-Instruct.Q5_K_M.gguf"
    )

# 2. Initialize the model on the CPU
print("🔌 Initializing Nemotron on Colab CPU Tier...")
llm = Llama(
    model_path=nemotron_path,
    n_ctx=2048,
    n_threads=2,
    verbose=False
)



In [ ]:
# use now
# Block-602
# 3. Define the POC targeted Code Review Script
# 1. Format using the strict Chat ML layout to force an 'instruction' context
chat_messages = [
    {"role": "system", "content": "You are an expert Python code reviewer. Identify critical runtime bugs, styling issues, and suggest robust corrections."},
    {"role": "user", "content": f"Perform a concise code review on this function:\n```python\ndef process(data):\n    x = data/0\n    return x\n```"}
]

# 2. Convert standard message layout into a raw prompt string
# llama-cpp-python automatically handles chat conversions behind the scenes:
print("\n🤖 Running strict instruction-based code review...")
response = llm.create_chat_completion(
    messages=chat_messages,
    max_tokens=512,
    temperature=0.1 # Dropped to 0.1 for sharp, exact debugging accuracy
)

print("\n=== Nemotron's High-Quality Code Review ===")
print(response["choices"][0]["message"]["content"])


